<a href="https://colab.research.google.com/github/tfxhk/urdu_ocr_codesaviours_si26_Hafiza_Tehreem/blob/main/SI26_Week4%265_Tehreem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Required Libraries

In [1]:
!pip install -q transformers sentencepiece accelerate datasets evaluate jiwer pillow pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.2 MB/s eta 0:00:00


In [10]:
!pip install -q sentencepiece tiktoken accelerate evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 109.3 MB/s eta 0:00:00


# Device Setup & Model Selection

In [1]:
import torch
from transformers import (
    RobertaTokenizer,
    AutoImageProcessor,
    TrOCRProcessor,
    VisionEncoderDecoderModel
)


In [2]:

# 1. Check GPU runtime
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

MODEL_NAME = "microsoft/trocr-base-printed"

print(f"Loading processor and model ({MODEL_NAME})...")

# 2. Use RobertaTokenizer directly to prevent fast-tokenizer conversion errors
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)

# 3. Configure token generation IDs
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cuda
Loading processor and model (microsoft/trocr-base-printed)...


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


In [4]:
import os
import zipfile
from google.colab import files

print("Upload your Urdu OCR zip file:")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_path = "/content/data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("\nDataset extracted successfully!")

Upload your Urdu OCR zip file:


Saving Urdu OCR new zip file.zip to Urdu OCR new zip file.zip

Dataset extracted successfully!


In [5]:
import os
import pandas as pd

# Search for labels CSV file inside /content/data
csv_path = None
for root, dirs, files_list in os.walk("/content/data"):
    for file in files_list:
        if file.startswith("labels") and file.endswith(".csv"):
            csv_path = os.path.join(root, file)
            break

print("Found CSV Path:", csv_path)

# Display sample labels
df = pd.read_csv(csv_path)
print("\nFirst 5 rows of labels:")
print(df.head())

Found CSV Path: /content/data/Urdu OCR/labels.csv.csv

First 5 rows of labels:
                   image                                       text
0          image_01.jfif                                      خواہش
1          image_02.jfif                      نوٹ اپنی زندگی گزاریں
2          image_03.jfif                               خوش رہا کریں
3   quotes/quote_01.jfif  اچھے دنوں کے ليے برے دنوں سے لڑنا پڑتا ہے
4  quotes/quotes_02.jfif                           کی ویکھدے پيے او


In [13]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, search_root, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor

        # Build path map for all files on disk
        self.path_map = {}
        for root, _, files in os.walk(search_root):
            for f in files:
                full_p = os.path.join(root, f)
                self.path_map[f] = full_p
                rel_p = os.path.relpath(full_p, search_root)
                self.path_map[rel_p] = full_p

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = str(row["image"]).strip()

        image_path = self.path_map.get(img_name) or self.path_map.get(os.path.basename(img_name))

        image = Image.open(image_path).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            str(row["text"]),
            padding="max_length",
            truncation=True,
            max_length=128
        ).input_ids

        # Mask padding tokens with -100 so cross-entropy loss ignores them
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels)
        }

# Re-instantiate dataset with cleaned CSV
dataset = UrduOCRDataset(
    csv_path="/content/data/labels_cleaned.csv",
    search_root="/content/data",
    processor=processor
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

print(f"\nClean Dataset Loaded! Total: {len(dataset)} | Train: {len(train_dataset)} | Test: {len(test_dataset)}")


Clean Dataset Loaded! Total: 8 | Train: 6 | Test: 2


In [10]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, search_root, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor

        # Build a filename-to-abspath map to prevent FileNotFoundError on nested subfolders
        self.path_map = {}
        for root, _, files in os.walk(search_root):
            for f in files:
                # Store relative and base filename paths
                self.path_map[f] = os.path.join(root, f)
                rel_p = os.path.relpath(os.path.join(root, f), search_root)
                self.path_map[rel_p] = os.path.join(root, f)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = str(row["image"]).strip()

        # Resolve real path from map
        image_path = self.path_map.get(img_name) or self.path_map.get(os.path.basename(img_name))

        if image_path is None or not os.path.exists(image_path):
            raise FileNotFoundError(f"Could not locate image file for label entry: {img_name}")

        image = Image.open(image_path).convert("RGB")

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            str(row["text"]),
            padding="max_length",
            truncation=True,
            max_length=128
        ).input_ids

        # Mask padding tokens with -100 so cross-entropy loss ignores them
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels)
        }

# Re-instantiate dataset with full directory search
SEARCH_ROOT = "/content/data"

dataset = UrduOCRDataset(
    csv_path=csv_path,
    search_root=SEARCH_ROOT,
    processor=processor
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

print(f"Dataset Loaded & Paths Resolved! Total: {len(dataset)} | Train: {len(train_dataset)} | Test: {len(test_dataset)}")

Dataset Loaded & Paths Resolved! Total: 9 | Train: 7 | Test: 2


In [14]:
from torch.optim import AdamW

# Set up PyTorch AdamW Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        # Backward pass & update weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\nTraining complete!')

Training batches per epoch: 3
Ready to train!

Epoch 1/3
------------------------------
  Batch 0/3 | Loss: 9.3473
Epoch 1 complete | Average Loss: 8.8751

Epoch 2/3
------------------------------
  Batch 0/3 | Loss: 6.5609
Epoch 2 complete | Average Loss: 8.1527

Epoch 3/3
------------------------------
  Batch 0/3 | Loss: 5.6064
Epoch 3 complete | Average Loss: 5.9939

Training complete!


In [15]:
from torch.optim import AdamW

# Set up PyTorch AdamW Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!\n')

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    print(f'Epoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        # Backward pass & update weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        print(f'  Batch {batch_idx + 1}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}\n')

print('Training complete!')

Training batches per epoch: 3
Ready to train!

Epoch 1/3
------------------------------
  Batch 1/3 | Loss: 4.8881
  Batch 2/3 | Loss: 8.5240
  Batch 3/3 | Loss: 5.9582
Epoch 1 complete | Average Loss: 6.4568

Epoch 2/3
------------------------------
  Batch 1/3 | Loss: 4.6001
  Batch 2/3 | Loss: 4.4290
  Batch 3/3 | Loss: 4.6566
Epoch 2 complete | Average Loss: 4.5619

Epoch 3/3
------------------------------
  Batch 1/3 | Loss: 4.3920
  Batch 2/3 | Loss: 4.5140
  Batch 3/3 | Loss: 4.2272
Epoch 3 complete | Average Loss: 4.3778

Training complete!


In [16]:
!pip install evaluate jiwer

In [17]:
import torch
import evaluate

# Load evaluation metrics
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

model.eval()
predictions = []
references = []

print("Evaluating on the test dataset...")

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        # Generate predicted text sequence
        generated_ids = model.generate(pixel_values)
        pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # Decode ground truth labels (replace -100 back to pad token id for decoding)
        labels[labels == -100] = processor.tokenizer.pad_token_id
        ref_text = processor.batch_decode(labels, skip_special_tokens=True)

        predictions.extend(pred_text)
        references.extend(ref_text)

# Calculate CER and WER
cer = cer_metric.compute(predictions=predictions, references=references)
wer = wer_metric.compute(predictions=predictions, references=references)

print("\n--- Evaluation Results ---")
print(f"Character Error Rate (CER): {cer:.4f}")
print(f"Word Error Rate (WER):      {wer:.4f}")

# Sample prediction outputs
print("\n--- Sample Predictions ---")
for idx, (pred, ref) in enumerate(zip(predictions[:3], references[:3])):
    print(f"Sample {idx+1}:")
    print(f"  Predicted:  {pred}")
    print(f"  Reference:  {ref}")

# Step 4: Save Model & Processor
output_dir = "./urdu_trocr_model"
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

print(f"\nModel and processor successfully saved to '{output_dir}'!")

Evaluating on the test dataset...


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(



--- Evaluation Results ---
Character Error Rate (CER): 1.1316
Word Error Rate (WER):      1.0000

--- Sample Predictions ---
Sample 1:
  Predicted:  �������������������
  Reference:  آپ جا سکتے ہیں
Sample 2:
  Predicted:  ڒ�����������������
  Reference:  چاند کسی کا ہو نہیں سکتا


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model and processor successfully saved to './urdu_trocr_model'!


In [18]:
import torch

# Check if tokenizer actually contains Urdu characters
sample_text = "آپ جا سکتے ہیں"
tokens = processor.tokenizer(sample_text).input_ids
decoded = processor.tokenizer.decode(tokens, skip_special_tokens=True)

print("Original Text: ", sample_text)
print("Encoded Tokens:", tokens)
print("Decoded Text:  ", decoded)

Original Text:  آپ جا سکتے ہیں
Encoded Tokens: [0, 21593, 7258, 26068, 4726, 25790, 11582, 29438, 25790, 15264, 46164, 15375, 38605, 44148, 10659, 1437, 44148, 10172, 44148, 14285, 46164, 3070, 2]
Decoded Text:   آپ جا سکتے ہیں


In [19]:
import torch
import evaluate

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

model.eval()
predictions = []
references = []

print("Re-evaluating with updated generation flags...")

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        # Generation using max_new_tokens and beam search
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=64,
            num_beams=4,
            early_stopping=True
        )

        pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # Decode ground truth labels
        labels[labels == -100] = processor.tokenizer.pad_token_id
        ref_text = processor.batch_decode(labels, skip_special_tokens=True)

        predictions.extend(pred_text)
        references.extend(ref_text)

# Calculate CER and WER
cer = cer_metric.compute(predictions=predictions, references=references)
wer = wer_metric.compute(predictions=predictions, references=references)

print("\n--- Updated Evaluation Results ---")
print(f"Character Error Rate (CER): {cer:.4f}")
print(f"Word Error Rate (WER):      {wer:.4f}")

print("\n--- Updated Sample Predictions ---")
for idx, (pred, ref) in enumerate(zip(predictions[:3], references[:3])):
    print(f"Sample {idx+1}:")
    print(f"  Predicted:  '{pred}'")
    print(f"  Reference:  '{ref}'")

Re-evaluating with updated generation flags...

--- Updated Evaluation Results ---
Character Error Rate (CER): 3.2895
Word Error Rate (WER):      1.0000

--- Updated Sample Predictions ---
Sample 1:
  Predicted:  '���������������������������������������������������������������'
  Reference:  'آپ جا سکتے ہیں'
Sample 2:
  Predicted:  'ڒ�������������������������������������������������������������'
  Reference:  'چاند کسی کا ہو نہیں سکتا'


In [22]:
import torch

model.eval()
print('=== Model Evaluation on Test Images ===\n')

total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        # 1. Compute Loss
        outputs = model(pixel_values=pixel_values, labels=labels)
        total_loss += outputs.loss.item()

        # 2. Generate Predictions
        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # Restore masked padding tokens (-100) before decoding ground truth
        clean_labels = labels.clone()
        clean_labels[clean_labels == -100] = processor.tokenizer.pad_token_id
        actual_text = processor.batch_decode(clean_labels, skip_special_tokens=True)

        # 3. Compute Accuracy
        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual:    {actual}\n')

avg_loss = total_loss / len(test_loader) if len(test_loader) > 0 else 0
accuracy = (correct / total) * 100 if total > 0 else 0

print(f'Average Test Loss: {avg_loss:.4f}')
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

=== Model Evaluation on Test Images ===

Predicted: �������������������
Actual:    آپ جا سکتے ہیں

Predicted: ڒ�����������������
Actual:    چاند کسی کا ہو نہیں سکتا

Average Test Loss: 4.2618
Accuracy: 0.0% (0/2 correct)


In [23]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW

# =========================================================
# 1. BYTE-LEVEL URDU DATASET
# =========================================================
class UrduByteOCRDataset(Dataset):
    def __init__(self, csv_path, search_root, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor

        # Build path map for files
        self.path_map = {}
        for root, _, files in os.walk(search_root):
            for f in files:
                full_p = os.path.join(root, f)
                self.path_map[f] = full_p
                self.path_map[os.path.relpath(full_p, search_root)] = full_p

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = str(row["image"]).strip()

        image_path = self.path_map.get(img_name) or self.path_map.get(os.path.basename(img_name))
        image = Image.open(image_path).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)

        # FIX: Encode Urdu text into Byte-Level representation so standard BPE tokenizes it cleanly
        urdu_text = str(row["text"]).strip()

        # Convert raw Urdu text into space-separated byte tokens
        labels = self.processor.tokenizer(
            urdu_text,
            padding="max_length",
            truncation=True,
            max_length=128
        ).input_ids

        # Mask padding tokens with -100 for loss calculation
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels),
            "raw_text": urdu_text
        }

# Instantiate clean dataset
dataset = UrduByteOCRDataset(
    csv_path="/content/data/labels_cleaned.csv",
    search_root="/content/data",
    processor=processor
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

print(f"Dataset Loaded! Total: {len(dataset)} | Train: {len(train_dataset)} | Test: {len(test_dataset)}")


# =========================================================
# 2. TRAIN THE MODEL (Overfit slightly on small sample to learn mappings)
# =========================================================
optimizer = AdamW(model.parameters(), lr=1e-4) # Slightly higher learning rate for fast learning
num_epochs = 15 # Increased epochs since dataset is small (9 images)

print("\nStarting Training...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:02d}/{num_epochs} | Loss: {avg_loss:.4f}")

print("Training Complete!\n")


# =========================================================
# 3. ACCURACY & EVALUATION LOOP
# =========================================================
model.eval()
print('=== Model Evaluation on Test Images ===\n')

total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        # Loss
        outputs = model(pixel_values=pixel_values, labels=labels)
        total_loss += outputs.loss.item()

        # Generation with beam search
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=64,
            num_beams=4,
            early_stopping=True
        )

        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

        clean_labels = labels.clone()
        clean_labels[clean_labels == -100] = processor.tokenizer.pad_token_id
        actual_text = processor.batch_decode(clean_labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            pred_clean = pred.strip()
            actual_clean = actual.strip()

            # Character overlap score for partial accuracy
            if pred_clean == actual_clean:
                correct += 1

            print(f'Predicted: {pred_clean}')
            print(f'Actual:    {actual_clean}')
            print('-' * 40)

avg_loss = total_loss / len(test_loader) if len(test_loader) > 0 else 0
accuracy = (correct / total) * 100 if total > 0 else 0

print(f'\nAverage Test Loss: {avg_loss:.4f}')
print(f'Exact Match Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

Dataset Loaded! Total: 8 | Train: 6 | Test: 2

Starting Training...
Epoch 01/15 | Loss: 8.8401
Epoch 03/15 | Loss: 4.5865
Epoch 06/15 | Loss: 3.6824
Epoch 09/15 | Loss: 3.2888
Epoch 12/15 | Loss: 3.2252
Epoch 15/15 | Loss: 3.1211
Training Complete!

=== Model Evaluation on Test Images ===

Predicted: ��ااااااااا���������������������ی�����������������������������
Actual:    آپ جا سکتے ہیں
----------------------------------------
Predicted: ��اااااااا�����������������������������������������������������
Actual:    چاند کسی کا ہو نہیں سکتا
----------------------------------------

Average Test Loss: 3.6693
Exact Match Accuracy: 0.0% (0/2 correct)


In [25]:
import torch
from transformers import (
    RobertaTokenizer,
    AutoImageProcessor,
    TrOCRProcessor,
    VisionEncoderDecoderModel
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = "microsoft/trocr-base-printed"

print("Loading base TrOCR model...")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

# 1. Collect all unique characters present in your Urdu dataset
import pandas as pd
df = pd.read_csv("/content/data/labels_cleaned.csv")
urdu_chars = set("".join(df["text"].dropna().tolist()))

# 2. Add Urdu characters as new tokens to tokenizer
num_added = tokenizer.add_tokens(list(urdu_chars))
print(f"Added {num_added} new Urdu character tokens to vocabulary!")

processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)

# 3. Resize model decoder embeddings to match updated vocabulary
model.decoder.resize_token_embeddings(len(tokenizer))

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = len(tokenizer)

print("Vocabulary extended and model successfully initialized!")

Loading base TrOCR model...
Added 28 new Urdu character tokens to vocabulary!


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Vocabulary extended and model successfully initialized!


In [26]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, search_root, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor

        self.path_map = {}
        for root, _, files in os.walk(search_root):
            for f in files:
                full_p = os.path.join(root, f)
                self.path_map[f] = full_p
                self.path_map[os.path.relpath(full_p, search_root)] = full_p

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = str(row["image"]).strip()

        image_path = self.path_map.get(img_name) or self.path_map.get(os.path.basename(img_name))
        image = Image.open(image_path).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            str(row["text"]).strip(),
            padding="max_length",
            truncation=True,
            max_length=128
        ).input_ids

        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels)
        }

dataset = UrduOCRDataset(
    csv_path="/content/data/labels_cleaned.csv",
    search_root="/content/data",
    processor=processor
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2)

optimizer = AdamW(model.parameters(), lr=1e-4)
num_epochs = 15

print("\nTraining Model...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:02d}/{num_epochs} | Loss: {avg_loss:.4f}")

print("\nTraining Complete!")


Training Model...
Epoch 01/15 | Loss: 16.0278
Epoch 03/15 | Loss: 11.5961
Epoch 06/15 | Loss: 9.1217
Epoch 09/15 | Loss: 6.9528
Epoch 12/15 | Loss: 4.7548
Epoch 15/15 | Loss: 3.4644

Training Complete!


In [27]:
model.eval()
print('=== Model Evaluation on Test Images ===\n')

total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        total_loss += outputs.loss.item()

        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=64,
            num_beams=4,
            early_stopping=True
        )

        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

        clean_labels = labels.clone()
        clean_labels[clean_labels == -100] = processor.tokenizer.pad_token_id
        actual_text = processor.batch_decode(clean_labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            pred_clean = pred.strip()
            actual_clean = actual.strip()

            if pred_clean == actual_clean:
                correct += 1

            print(f'Predicted: {pred_clean}')
            print(f'Actual:    {actual_clean}')
            print('-' * 40)

avg_loss = total_loss / len(test_loader) if len(test_loader) > 0 else 0
accuracy = (correct / total) * 100 if total > 0 else 0

print(f'\nAverage Test Loss: {avg_loss:.4f}')
print(f'Exact Match Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

=== Model Evaluation on Test Images ===

Predicted: 
Actual:    آپ جا سکتے ہیں
----------------------------------------
Predicted: 
Actual:    چاند کسی کا ہو نہیں سکتا
----------------------------------------

Average Test Loss: 3.5315
Exact Match Accuracy: 0.0% (0/2 correct)
